# Mi primer LLM CHAT

Bienvenido a tu primer experimento con **modelos de lenguaje locales** usando **llama.cpp**.

En este notebook aprenderás a:
- Entender qué es `llama.cpp` y por qué es tan relevante
- Instalar la librería `llama-cpp-python`
- Descargar un modelo cuantizado desde Hugging Face
- Cargar el modelo en memoria
- Generar texto con diferentes configuraciones
- Entender cada parámetro de generación
- Usar el formato de **chat** (mensajes de sistema + usuario)
- Implementar **streaming** de respuestas

> **Nota de clase:** No necesitás internet en tiempo de ejecución ni GPU costosa.  
> Todo corre **100% local y offline** gracias a `llama.cpp`.

---
## 1. ¿Qué es llama.cpp?

`llama.cpp` es una librería escrita en **C/C++** que permite correr modelos de lenguaje grande (LLMs) 
directamente en tu ordenador, sin necesidad de GPU de alto coste ni servicios en la nube.

### ¿Por qué es tan importante?

| Característica | Ventaja |
|---|---|
| **CPU-first** | Funciona en cualquier ordenador, incluso sin GPU |
| **Metal (Apple Silicon)** | Aprovecha la GPU de los Mac M1/M2/M3/M4 automáticamente |
| **Cuantización (GGUF)** | Reduce el modelo hasta 4 bits, de 30 GB → ~4 GB |
| **Offline** | Sin conexión, sin costes de API, privacidad total |
| **Velocidad** | Inferencia en segundos a minutos en hardware moderno |

### El formato GGUF

Los modelos que usa llama.cpp están en formato **GGUF** (sucesor de GGML).  
Este formato incluye **cuantización**, que es comprimir el modelo reduciendo la precisión de los pesos:

```
Q4_K_M  → 4 bits, calidad media  (~2.5 GB para un modelo 3B params)
Q8_0    → 8 bits, alta calidad   (~3.5 GB para un modelo 3B params)
F16     → sin cuantizar, máxima  (~6 GB para un modelo 3B params)
```

> **Regla general:** `Q4_K_M` ofrece el mejor balance velocidad/calidad para experimentar.

---
## 2. Instalación de llama-cpp-python

`llama-cpp-python` es el binding oficial de Python para `llama.cpp`.

### Opciones de instalación según tu hardware

**Opción A – Mac Apple Silicon (M1/M2/M3/M4) → Metal GPU** ✅ recomendado si estás en este Mac
```bash
CMAKE_ARGS="-DGGML_METAL=on" pip install llama-cpp-python --upgrade
```

**Opción B – PC con GPU NVIDIA → CUDA**
```bash
CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade
```

**Opción C – Solo CPU (cualquier máquina)**
```bash
pip install llama-cpp-python --upgrade
```

> ⚠️ La instalación tarda varios minutos porque **compila código C++** en tu máquina.  
> Verás mucha salida de compilación — eso es completamente normal.

**La celda de código siguiente detecta tu plataforma y elige la opción correcta automáticamente.**

In [1]:
import platform
import subprocess
import sys
import os

# Detectar plataforma
sistema     = platform.system()
arquitectura = platform.machine()

print(f"Sistema operativo : {sistema}")
print(f"Arquitectura      : {arquitectura}")
print(f"Python            : {sys.version.split()[0]}")
print()

# Elegir flags de compilación según plataforma
if sistema == "Darwin" and arquitectura == "arm64":
    print("✅ Apple Silicon detectado → compilando con Metal (GPU integrada)")
    cmake_args = "-DGGML_METAL=on"
else:
    print("ℹ️  Instalando versión solo-CPU")
    cmake_args = ""

env = {**os.environ}
if cmake_args:
    env["CMAKE_ARGS"] = cmake_args

# Instalar / actualizar llama-cpp-python
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "llama-cpp-python", "--upgrade", "--quiet"],
    env=env,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ llama-cpp-python instalado/actualizado correctamente")
else:
    print("❌ Error al instalar:")
    print(result.stderr)

Sistema operativo : Darwin
Arquitectura      : arm64
Python            : 3.12.3

✅ Apple Silicon detectado → compilando con Metal (GPU integrada)
✅ llama-cpp-python instalado/actualizado correctamente


---
## 3. Verificar la instalación

Verificamos que `llama-cpp-python` importa sin errores y comprueba la versión instalada.

In [2]:
try:
    import llama_cpp
    print(f"✅ llama_cpp importado correctamente")
    print(f"   Versión: {llama_cpp.__version__}")
except ImportError as e:
    print(f"❌ Error al importar llama_cpp: {e}")
    print("   Reinicia el kernel y vuelve a ejecutar la celda de instalación")
except Exception as e:
    print(f"⚠️  Advertencia: {e}")

✅ llama_cpp importado correctamente
   Versión: 0.3.19


---
## 4. Descargar un modelo GGUF desde Hugging Face

Vamos a descargar un modelo pequeño pero capaz: **Qwen2.5-0.5B-Instruct** en formato `Q4_K_M`.

### ¿Por qué este modelo para empezar?

| Característica | Valor |
|---|---|
| Parámetros | 500 millones (0.5B) |
| Tamaño descargado | ~350 MB |
| RAM necesaria | ~500 MB |
| Velocidad (M1) | ~30-80 tokens/segundo |
| Calidad | Suficiente para demostración y aprendizaje |

> 📦 Usaremos `huggingface_hub` para descargar el archivo GGUF directamente.  
> El modelo se guarda en caché local en `~/.cache/huggingface/` para no volver a descargarse.

### ¿Cómo buscar modelos GGUF?

1. Ve a [huggingface.co](https://huggingface.co)
2. Busca el modelo que quieras (ej: `Llama-3.2-1B-Instruct`)
3. Filtra por formato **GGUF** o busca el repositorio `bartowski` que empaqueta muchos modelos
4. Copia el `repo_id` y el nombre del archivo `.gguf`

In [3]:
from huggingface_hub import hf_hub_download
import os

# ─── Configuración del modelo ────────────────────────────────────────────────
# Podés cambiar estos valores para probar otros modelos
REPO_ID   = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
FILENAME  = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
# ─────────────────────────────────────────────────────────────────────────────

print(f"📥 Descargando modelo...")
print(f"   Repositorio : {REPO_ID}")
print(f"   Archivo     : {FILENAME}")
print(f"   Dest.       : ~/.cache/huggingface/ (caché automático)")
print()

# hf_hub_download descarga el archivo si no existe en caché
# y devuelve la ruta local del archivo descargado
ruta_modelo = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
)

tamaño_mb = os.path.getsize(ruta_modelo) / (1024 * 1024)

print(f"✅ Modelo listo en:")
print(f"   {ruta_modelo}")
print(f"   Tamaño: {tamaño_mb:.1f} MB")

📥 Descargando modelo...
   Repositorio : Qwen/Qwen2.5-0.5B-Instruct-GGUF
   Archivo     : qwen2.5-0.5b-instruct-q4_k_m.gguf
   Dest.       : ~/.cache/huggingface/ (caché automático)



/Users/inti/GitHub/model_train/model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Modelo listo en:
   /Users/inti/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf
   Tamaño: 468.6 MB


---
## 5. Cargar el modelo en memoria

Ahora cargamos el modelo con `llama_cpp.Llama`. Este paso puede tardar **5-20 segundos** la primera vez.

### Parámetros clave de `Llama()`

| Parámetro | Qué hace | Ejemplo |
|---|---|---|
| `model_path` | Ruta al archivo `.gguf` | `"/ruta/al/modelo.gguf"` |
| `n_ctx` | Tamaño de la **ventana de contexto** en tokens | `2048` = recuerda ~1500 palabras |
| `n_gpu_layers` | Capas movidas a GPU (**-1 = todas**) | `-1` para máxima velocidad |
| `verbose` | Mostrar logs internos de llama.cpp | `False` para silenciar |

### ¿Qué es `n_ctx` (context window)?

Es cuántos tokens "recuerda" el modelo a la vez. Incluye tanto el prompt como la respuesta generada.

```
1 token ≈ 0.75 palabras en español
2048 tokens ≈ ~1500 palabras ≈ una página A4
4096 tokens ≈ ~3000 palabras ≈ un artículo corto
```

> ⚠️ Más contexto = más RAM usada. Con modelos pequeños, 2048 es suficiente para practicar.

In [4]:
from llama_cpp import Llama

print("⏳ Cargando el modelo en memoria...")
print("   (primera carga: 5-20 seg según tu hardware)\n")

llm = Llama(
    model_path  = ruta_modelo,   # ruta local del archivo .gguf descargado antes
    n_ctx       = 2048,          # ventana de contexto: 2048 tokens (~1500 palabras)
    n_gpu_layers = -1,           # -1 = cargar TODAS las capas en GPU (Metal/CUDA)
                                 #  0 = solo CPU
    verbose     = False,         # silenciar logs internos de llama.cpp
)

print("✅ Modelo cargado exitosamente")
print(f"   Contexto máximo : {llm.n_ctx()} tokens")
print(f"   Vocabulario     : {llm.n_vocab()} tokens")

⏳ Cargando el modelo en memoria...
   (primera carga: 5-20 seg según tu hardware)



llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Modelo cargado exitosamente
   Contexto máximo : 2048 tokens
   Vocabulario     : 151936 tokens


---
## 6. Primera inferencia: generación de texto

¡Vamos a hacer nuestra primera llamada al modelo!

### ¿Cómo funciona la generación de texto?

El modelo recibe un texto de entrada (**prompt**) y predice, token a token, cuál es la continuación más probable.

```
Prompt → [Modelo] → Token 1 → Token 2 → Token 3 → ...
```

> 💡 No es magia: el modelo es esencialmente una función matemática que predice el siguiente token más probable dado el texto anterior.

In [ ]:
# Generación de texto más simple posible: "completar" un texto

# prompt = str(input("🖋️ Escribí un prompt para el modelo: "))

In [18]:
# Generación de texto más simple posible: "completar" un texto

prompt = "La inteligencia artificial es"

print(f"📝 Prompt: '{prompt}'")
print(f"{'─'*50}")

# llm() llama directamente al modelo en modo "completar texto"
respuesta = llm(
    prompt,
    max_tokens = 80,    # máximo 80 tokens en la respuesta
    echo       = True,  # incluir el prompt en la respuesta (ver contexto completo)
)

# La respuesta es un diccionario con varios campos
texto_completo = respuesta["choices"][0]["text"]
print(texto_completo)

print(f"\n{'─'*50}")
print(f"📊 Estadísticas:")
print(f"   Tokens en prompt   : {respuesta['usage']['prompt_tokens']}")
print(f"   Tokens generados   : {respuesta['usage']['completion_tokens']}")
print(f"   Tokens totales     : {respuesta['usage']['total_tokens']}")

📝 Prompt: 'La inteligencia artificial es'
──────────────────────────────────────────────────
La inteligencia artificial es una tecnología que ha tenido un impacto significativo en nuestra vida diaria y en la sociedad en general. La inteligencia artificial es un tipo de inteligencia artificial en la que la inteligencia computacional se utiliza para realizar tareas de manera automática y consistente. Como en un juego de adivinanzas, el aprendiz está en constante cambio y se está adaptando

──────────────────────────────────────────────────
📊 Estadísticas:
   Tokens en prompt   : 5
   Tokens generados   : 80
   Tokens totales     : 85


---
## 7. Explorando los parámetros de generación

Los parámetros controlan **cómo** el modelo elige cada token. Son fundamentales para entender el comportamiento.

### Los 3 parámetros más importantes

#### 🌡️ `temperature` — Creatividad vs. precisión

Controla qué tan "arriesgadas" son las elecciones de tokens.

```
temperature = 0.0  →  Siempre elige el token más probable (determinista, repetitivo)
temperature = 0.7  →  Balance razonable (recomendado para tareas generales)
temperature = 1.0  →  Distribución original del modelo
temperature = 2.0  →  Muy aleatorio, puede volverse incoherente
```

#### 🔪 `top_p` — Nucleus sampling

Solo considera los tokens que juntos suman `top_p` de probabilidad.

```
top_p = 1.0   →  Considera TODOS los tokens (sin filtro)
top_p = 0.95  →  Considera el núcleo del 95% de probabilidad (buen default)
top_p = 0.5   →  Solo los tokens más probables (más conservador)
```

#### 🔝 `top_k` — Limitar el vocabulario de elección

Solo considera los `k` tokens más probables en cada paso.

```
top_k = 0   →  Sin límite (desactivado)
top_k = 40  →  Típico: considera solo los 40 tokens más probables
top_k = 1   →  Equivalente a greedy (siempre el más probable)
```

> 📌 En la práctica, se usan `temperature` + `top_p` juntos. Es raro usar los tres a la vez.

In [19]:
# Experimentar con diferentes valores de temperature
prompt_test = "Los robots del futuro van a"

print("🔬 Experimento: mismo prompt, distinta temperatura\n")
print(f"Prompt: '{prompt_test}'\n")

configuraciones = [
    {"temperature": 0.0, "label": "Frío (determinista)"},
    {"temperature": 0.7, "label": "Templado (recomendado)"},
    {"temperature": 1.5, "label": "Caliente (muy creativo)"},
]

for config in configuraciones:
    resultado = llm(
        prompt_test,
        max_tokens  = 50,
        temperature = config["temperature"],
        top_p       = 0.95,
        echo        = False,   # no repetir el prompt en la salida
    )
    texto = resultado["choices"][0]["text"].strip()
    print(f"🌡️  temp={config['temperature']} ({config['label']})")
    print(f"   {texto}")
    print()

🔬 Experimento: mismo prompt, distinta temperatura

Prompt: 'Los robots del futuro van a'

🌡️  temp=0.0 (Frío (determinista))
   ser más inteligentes y inteligentes que los humanos. ¿Qué es lo que los robots van a hacer? - El Mundo - La Nación
El futuro de los robots es un tema que ha tenido un gran impacto en la indust

🌡️  temp=0.7 (Templado (recomendado))
   ser las herramientas más importantes del mundo de la inteligencia artificial, pero la inteligencia humana es el verdadero piloto del futuro.
La inteligencia humana se caracteriza por la capacidad de entender y procesar información. La inteligencia

🌡️  temp=1.5 (Caliente (muy creativo))
   convertirse en el corazón del desarrollo industrial del mañana. How do you say this sentence in English?

The robots of the future will revolutionize the future of industrial development. The robots of the future, which will revolutionize the industrial development of the future



---
## 8. Prompt Engineering: cómo hablarle al modelo

El **prompt** es el texto que le das al modelo como entrada. Su estructura influye enormemente en la calidad de la respuesta.

### Técnicas básicas

#### 1. Instrucción directa
```
"Traduce al inglés: el cielo es azul"
```

#### 2. Rol + Instrucción
```
"Eres un profesor experto en física. Explica qué es un agujero negro en 3 frases simples."
```

#### 3. Few-shot (dar ejemplos)
```
"Clasifica el sentimiento como POSITIVO o NEGATIVO:
Texto: Me encanta este producto → POSITIVO
Texto: Qué mala experiencia → NEGATIVO
Texto: No volvería a comprarlo → "
```

#### 4. Chain-of-thought (paso a paso)
```
"Resuelve esto paso a paso: Si tengo 5 manzanas y compro 3 más..."
```

> 💡 **Regla de oro del prompting:** Sé específico sobre el formato, longitud y estilo de la respuesta que esperás.

In [27]:
# Comparar: prompt simple vs. prompt con instrucción clara
print("🔬 Comparación de calidad de prompts\n")
print("=" * 60)

# ── Prompt 1: vago ─────────────────────────────────────────────
prompt_vago = "Python"
print(f"\n❌ Prompt vago: '{prompt_vago}'")
resultado_vago = llm(prompt_vago, max_tokens=60, temperature=0.7, echo=False)
print(f"→ {resultado_vago['choices'][0]['text'].strip()}")

# ── Prompt 2: con instrucción clara ────────────────────────────
prompt_claro = (
    "Explica qué es Python en exactamente 2 frases "
    "para alguien que nunca programó:"
)
print(f"\n✅ Prompt claro: '{prompt_claro}'")
resultado_claro = llm(prompt_claro, max_tokens=80, temperature=0.3, echo=False)
print(f"→ {resultado_claro['choices'][0]['text'].strip()}")

# ── Prompt 3: few-shot ─────────────────────────────────────────
prompt_fewshot = (
    "Clasifica el sentimiento de estas frases:\n"
    "Frase: 'Me encanta el café por la mañana' → POSITIVO\n"
    "Frase: 'El tráfico de hoy fue horrible' → NEGATIVO\n"
    "Frase: 'Hoy aprendí algo nuevo' →"
)
print(f"\n🎯 Few-shot prompt:")
resultado_fewshot = llm(prompt_fewshot, max_tokens=4, temperature=0.1, echo=False)
print(f"→ {resultado_fewshot['choices'][0]['text'].strip()}")

🔬 Comparación de calidad de prompts


❌ Prompt vago: 'Python'
→ code for a simple calculator that adds two numbers and returns the sum.

Sure, here's a simple calculator function in Python that takes two numbers as input and returns their sum:

```python
def add_numbers(a, b):
    return a + b
```

You can call this function by passing

✅ Prompt claro: 'Explica qué es Python en exactamente 2 frases para alguien que nunca programó:'
→ Python es una lenguaje de programación que se utiliza para crear aplicaciones web y para programar en Python, es un lenguaje de programación que se utiliza para crear aplicaciones web y para programar en Python, es un lenguaje de programación que se utiliza para crear aplicaciones web y para programar en Python, es un lenguaje de programación que se

🎯 Few-shot prompt:
→ POSITIVO


---
## 9. Chat Completions: modo conversacional

Los modelos modernos tipo "Instruct" o "Chat" están entrenados para recibir mensajes con **roles** en lugar de texto plano.

### Los tres roles

| Rol | Descripción | Ejemplo |
|---|---|---|
| `system` | Instrucciones globales al modelo (personalidad, contexto, límites) | `"Eres un asistente amigable..."` |
| `user` | Lo que dice el usuario | `"¿Qué es el machine learning?"` |
| `assistant` | Lo que responde el modelo | _(generado automáticamente)_ |

### El formato de chat internamente

El modelo no "entiende" los roles como tal — el código convierte la lista de mensajes en un texto formateado según la plantilla del modelo.

Por ejemplo, Qwen2.5 usa:
```
<|im_start|>system
Eres un asistente útil.
<|im_end|>
<|im_start|>user
¿Hola?
<|im_end|>
<|im_start|>assistant
```

> 📌 Usar `create_chat_completion()` en vez de `__call__()` es la forma recomendada para modelos Instruct/Chat.

In [28]:
# Chat Completions: la forma correcta de usar modelos Instruct/Chat
print("💬 Chat Completions — modo conversacional\n")

mensajes = [
    {
        "role": "system",
        "content": (
            "Eres un profesor de programación amigable y claro. "
            "Explicas conceptos de manera simple usando analogías de la vida cotidiana. "
            "Tus respuestas son concisas: máximo 3 oraciones."
        )
    },
    {
        "role": "user",
        "content": "¿Qué es un algoritmo? Explícalo como si tuviese 10 años."
    }
]

print("📨 Mensajes enviados:")
for msg in mensajes:
    emoji = "⚙️" if msg["role"] == "system" else "👤"
    print(f"  {emoji} [{msg['role'].upper()}]: {msg['content'][:60]}...")

print(f"\n{'─'*50}")
print("🤖 Respuesta del modelo:\n")

respuesta_chat = llm.create_chat_completion(
    messages    = mensajes,
    max_tokens  = 150,
    temperature = 0.7,
    top_p       = 0.95,
)

# La respuesta tiene la misma estructura que la API de OpenAI
mensaje_respuesta = respuesta_chat["choices"][0]["message"]
print(f"{mensaje_respuesta['content']}")

print(f"\n{'─'*50}")
print(f"📊 Uso de tokens:")
uso = respuesta_chat["usage"]
print(f"   Prompt     : {uso['prompt_tokens']} tokens")
print(f"   Respuesta  : {uso['completion_tokens']} tokens")
print(f"   Total      : {uso['total_tokens']} tokens")

💬 Chat Completions — modo conversacional

📨 Mensajes enviados:
  ⚙️ [SYSTEM]: Eres un profesor de programación amigable y claro. Explicas ...
  👤 [USER]: ¿Qué es un algoritmo? Explícalo como si tuviese 10 años....

──────────────────────────────────────────────────
🤖 Respuesta del modelo:

Un algoritmo es como un juego que te ayuda a crear un juego divertido y fácil de jugar. Es como un juego de palabras donde te ayuda a aprender a escribir y a pensar. Es como un juego de inventar y crear para que te hagan galo con los nombres y el cómo.

──────────────────────────────────────────────────
📊 Uso de tokens:
   Prompt     : 79 tokens
   Respuesta  : 63 tokens
   Total      : 142 tokens


In [29]:
# Conversación multi-turno: simular un diálogo de ida y vuelta
print("🔄 Conversación multi-turno\n")

# Historial de mensajes — se va acumulando
historial = [
    {
        "role": "system",
        "content": "Eres un asistente experto en inteligencia artificial. Respuestas breves y precisas."
    }
]

# Preguntas del "usuario"
preguntas_usuario = [
    "¿Cuál es la diferencia entre IA, machine learning y deep learning?",
    "¿Puedes darme un ejemplo concreto de deep learning en la vida real?",
]

for pregunta in preguntas_usuario:
    # Agregar el turno del usuario al historial
    historial.append({"role": "user", "content": pregunta})

    print(f"👤 Usuario: {pregunta}")

    # Llamar al modelo con TODO el historial
    respuesta = llm.create_chat_completion(
        messages    = historial,
        max_tokens  = 120,
        temperature = 0.5,
    )

    respuesta_texto = respuesta["choices"][0]["message"]["content"]

    # Agregar la respuesta del asistente al historial (para el próximo turno)
    historial.append({"role": "assistant", "content": respuesta_texto})

    print(f"🤖 Asistente: {respuesta_texto}")
    print(f"{'─'*60}\n")

print(f"📚 El historial acumulado tiene {len(historial)} mensajes.")

🔄 Conversación multi-turno

👤 Usuario: ¿Cuál es la diferencia entre IA, machine learning y deep learning?
🤖 Asistente: La IA, Machine Learning y Deep Learning son términos que se utilizan para describir diferentes asuntos de inteligencia artificial y aprendizaje automático:

1. IA: Incluye la inteligencia artificial generalizada, que es la capacidad de un sistema para aprender y adaptarse a nuevas situaciones y datos. Es un campo de estudio y aplicaciones que se utiliza para crear sistemas inteligentes que pueden aprender y mejorar sus propias funciones.

2. Machine Learning: Es un tipo de IA que se utiliza para crear sistemas que pueden aprender a realizar tareas sin ser programados específ
────────────────────────────────────────────────────────────

👤 Usuario: ¿Puedes darme un ejemplo concreto de deep learning en la vida real?
🤖 Asistente: Claro, aquí tienes un ejemplo concreto de deep learning aplicado en la vida real:

En la industria de la construcción, un programa de deep learni

---
## 10. Streaming: ver la respuesta en tiempo real

Por defecto, el modelo genera **todos los tokens** y devuelve el texto completo al final.  
Con **streaming** activado, cada token se recibe y se puede mostrar **a medida que se genera**, como en ChatGPT.

### ¿Cuándo usar streaming?

| Situación | Recomendación |
|---|---|
| Interfaz de usuario (chatbot) | ✅ Usar streaming — mejor UX, el usuario ve la respuesta mientras se genera |
| Procesamiento batch (muchas consultas) | ❌ Sin streaming — más eficiente esperar el resultado completo |
| Respuestas cortas | ❌ Sin streaming — no se nota la diferencia |
| Respuestas largas (> 100 tokens) | ✅ Usar streaming — evita la espera |

> 💡 Técnicamente, el streaming no es más rápido. El modelo genera los tokens al mismo ritmo.  
> La diferencia es **cuándo se le muestran** al usuario.

In [30]:
import sys
import time

print("🌊 Streaming de respuesta (token a token)\n")
print("─" * 50)
print("🤖 ", end="", flush=True)

mensajes_stream = [
    {"role": "system", "content": "Eres un poeta. Responde siempre con un poem corto de 4 versos."},
    {"role": "user",   "content": "Escribe un poema sobre los modelos de lenguaje locales."}
]

contador_tokens = 0
texto_completo  = ""

# stream=True devuelve un generador de chunks en vez del resultado completo
generador = llm.create_chat_completion(
    messages    = mensajes_stream,
    max_tokens  = 120,
    temperature = 0.8,
    stream      = True,   # ← activar streaming
)

for chunk in generador:
    # Cada chunk tiene la misma estructura que la respuesta normal
    delta = chunk["choices"][0].get("delta", {})
    token_texto = delta.get("content", "")

    if token_texto:
        print(token_texto, end="", flush=True)  # imprimir sin salto de línea
        texto_completo += token_texto
        contador_tokens += 1
        time.sleep(0.01)  # pausa visual para apreciar el streaming

print(f"\n\n{'─'*50}")
print(f"✅ Streaming completado: ~{contador_tokens} tokens generados")

🌊 Streaming de respuesta (token a token)

──────────────────────────────────────────────────
🤖 En los ríos de mi ciudad, donde el agua fluye,  
Se forman los oraciones, con palabras y enunciados,  
Más de un millón de versos, que no me resbalan,  
En el teatro de mi mente, donde la historia se lanza.

Los modelos de lenguaje, el sistema de escritura,  
Son como los ríos de mi ciudad, con tinta, que fluye,  
Más de un millón de palabras, que no me resbalan,  
En mi mente, donde el

──────────────────────────────────────────────────
✅ Streaming completado: ~120 tokens generados


---
## 11. Bonus: medir el rendimiento

Una métrica clave al trabajar con LLMs locales es **tokens por segundo** (tokens/s).  
Nos dice qué tan rápido genera el modelo.

### Referencias de velocidad orientativas

| Hardware | Esperado para modelo 0.5B |
|---|---|
| Mac M1/M2 (Metal) | 80–150 tokens/s |
| Mac M3/M4 (Metal) | 150–300 tokens/s |
| Solo CPU moderna   | 20–60 tokens/s |
| rtx 3080 (CUDA)    | 200–400 tokens/s |

In [31]:
import time

print("⏱️  Midiendo rendimiento del modelo...\n")

prompt_benchmark = "Explica el concepto de inteligencia artificial en 100 palabras:"

inicio = time.perf_counter()

resultado_bench = llm.create_chat_completion(
    messages = [
        {"role": "system", "content": "Eres un asistente conciso."},
        {"role": "user",   "content": prompt_benchmark}
    ],
    max_tokens  = 100,
    temperature = 0.3,
    stream      = False,
)

fin = time.perf_counter()

# Calcular métricas
segundos        = fin - inicio
tokens_entrada  = resultado_bench["usage"]["prompt_tokens"]
tokens_salida   = resultado_bench["usage"]["completion_tokens"]
tokens_por_seg  = tokens_salida / segundos

print(f"{'─'*50}")
print(f"📊 Resultado del benchmark:")
print(f"   Tokens de entrada  : {tokens_entrada}")
print(f"   Tokens generados   : {tokens_salida}")
print(f"   Tiempo total       : {segundos:.2f} segundos")
print(f"   Velocidad          : {tokens_por_seg:.1f} tokens/segundo")
print(f"{'─'*50}")
print()
print("💬 Respuesta generada:")
print(resultado_bench["choices"][0]["message"]["content"])

⏱️  Midiendo rendimiento del modelo...

──────────────────────────────────────────────────
📊 Resultado del benchmark:
   Tokens de entrada  : 38
   Tokens generados   : 86
   Tiempo total       : 0.69 segundos
   Velocidad          : 124.6 tokens/segundo
──────────────────────────────────────────────────

💬 Respuesta generada:
La inteligencia artificial es un campo de investigación y desarrollo que se enfoca en crear sistemas que simulan la inteligencia humana para realizar tareas y procesos de manera más eficiente y rápida. Este tipo de tecnología se utiliza en diversas aplicaciones, como la inteligencia de IA en el comercio, la inteligencia artificial en el cuidado de la salud, y la inteligencia artificial en la inteligencia artificial.


---
## 12. Conclusiones y próximos pasos

### Lo que aprendiste en este notebook

✅ **llama.cpp** permite correr LLMs 100% local, sin cloud ni GPU costosa  
✅ Formato **GGUF** + cuantización = modelos potentes en hardware modesto  
✅ `hf_hub_download` descarga modelos desde Hugging Face con caché automático  
✅ `Llama()` carga el modelo con `n_ctx`, `n_gpu_layers` y `verbose`  
✅ `temperature`, `top_p`, `top_k` controlan creatividad vs. precisión  
✅ `create_chat_completion()` es la API recomendada para modelos Instruct  
✅ **Streaming** mejora la UX mostrando tokens a medida que se generan  
✅ Podés medir rendimiento con `tokens/segundo`

---

### Próximos pasos sugeridos

| Paso | Descripción |
|---|---|
| 🔬 Probar otros modelos | `Llama-3.2-1B-Instruct`, `Phi-3.5-mini`, `Mistral-7B-Instruct` |
| 🏗️ Fine-tuning básico | Entrenar un modelo base con tus propios datos (próximo notebook) |
| 🔗 RAG | Conectar el LLM con una base de documentos para responder preguntas |
| 🌐 API REST | Usar `llama-cpp-python[server]` para exponer el modelo como API HTTP |
| 📱 Producción | Empaquetarlo con FastAPI o integrarlo en una app web |

---

### Recursos recomendados

- 📂 [Hugging Face GGUF models](https://huggingface.co/models?library=gguf) — biblioteca de modelos en formato GGUF  
- 📖 [llama-cpp-python docs](https://llama-cpp-python.readthedocs.io/) — documentación oficial  
- 🧪 [LM Studio](https://lmstudio.ai/) — interfaz visual para explorar modelos locales  

> 🎯 **Tarea práctica:** Cambiá el `system` prompt para convertir el modelo en un  
> asistente especializado en tu tema favorito (historia, cocina, código, etc.)  
> y probá cómo cambia el comportamiento.